**Importing necessary libraries**

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from scipy import stats
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

**Importing the data set to a Pandas DataFrame**

In [ ]:
file_path = r'data_wrangling\final_data\historical_data.csv'
df = pd.read_csv(file_path)
df = df.drop(columns=['Unnamed: 0.1','Unnamed: 0'])

**sample of the dataframe**

In [ ]:
df.sample(5)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.columns

## **Univariate Analysis** 

**Numerical Variables**

In [ ]:
numerical_variables = ['year','car-age','fiscal-power','price','quarter','month']
num_df = df[numerical_variables]

*A general overview on each variable*

In [ ]:
fig, axes = plt.subplots(1, len(num_df.columns), figsize=(18.75,6))
for i, col in enumerate(num_df.columns):
    sns.boxplot(y=num_df[col], ax=axes[i], color='royalblue')
    axes[i].set_title(col)

plt.suptitle('Boxplot for Numerical Values', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(num_df.columns), figsize=(18.75,6))
for i, col in enumerate(num_df.columns):
    sns.histplot(num_df[col], ax = axes[i], bins=40, color='royalblue')
    axes[i].set_title(col)

plt.suptitle('Histograms for Numerical Values', fontsize=14)
plt.tight_layout()
plt.show()

* The *Year* variable is bimodal, indicating that most car listings were published in 2016 and earlier in the observed period, with a peak at 2009
* The *Car Age* distribution shows that, at the time of publication, most cars were less than 10 years old, though several outliers are present.
* The *Fiscal Power* variable is highly right-skewed, suggesting that most used cars fall within the range of 4 to 6 fiscal horsepower.
* The *Price* distribution indicates that the vast majority of vehicles were listed below **33,825 TND** between **2007** and **2025.**
* The *Quarter* distribution shows that most used cars are sold in the **third quarter**, with **September** recording the highest sales and **December** the lowest.

##### *Categorical Variables*

In [ ]:
categorical_variables = ['brand','model','fuel','location','body-type']
cat_df = df[categorical_variables]

*A general overview on each variable*

In [ ]:
fig, axes = plt.subplots(1, len(cat_df.columns)-1, figsize=(18.75,6))

sns.barplot(data=cat_df['brand'].value_counts().head(12), color='royalblue', ax = axes[0])
axes[0].set_title('Brand')
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(data=cat_df['model'].value_counts().head(12), color='royalblue', ax = axes[1])
axes[1].set_title('Model')
axes[1].tick_params(axis='x', rotation=45)

sns.barplot(data=cat_df['fuel'].value_counts(), color='royalblue', ax=axes[2])
axes[2].set_title('Fuel')
axes[2].tick_params(axis='x', rotation=45)

sns.barplot(data=df['body-type'].value_counts(), color='royalblue', ax=axes[3])
axes[3].set_title('Body Type')
axes[3].tick_params(axis='x', rotation=45)

plt.suptitle('Barplots for Categorical Variables')
plt.tight_layout()
plt.show()

* The *Brand* variable indicates that **German** and **French** car brands have dominated the used car market over the analyzed period.
* Popular *Models* such as the **Golf**, **Clio**, and **Polo** lead the market, likely reflecting the purchasing power of the average Tunisian consumer.
* **Petrol** vehicles account for roughly two-thirds of the market, while **diesel** cars make up the remaining share.
* **Citadine** *body type* has dominated the market, followed by *compacts* and *berlines*.

In [ ]:
fig = plt.figure(figsize=(8,6))
sns.barplot(data=cat_df['location'].value_counts().head(12), palette='tab20')
plt.xticks(rotation=45)
plt.xlabel('Location')
plt.ylabel('Cars per Location')
plt.title('Top Locations With the Most Number of Cars', fontsize=14)
plt.show()

* **Tunis** dominates the used car listings, followed by **Ariana** and **Ben Arous**, highlighting the Greater Tunis area as the main hub of *automotive activity*.

##### *Missing Values Overview*

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]

plt.figure(figsize=(6, 6))
sns.barplot(x=missing.values, y=missing.index, palette="viridis")
plt.title("Missing Values per Column")
plt.xlabel("Number of Missing Values")
plt.ylabel("Columns")
plt.show()

##### *The Yearly Frequency of Our Dataset*

In [ ]:
yearly = df['year'].value_counts().sort_index().reset_index()
yearly.columns = ['year', 'count']
avg = yearly['count'].mean()

plt.figure(figsize=(7,4))
sns.lineplot(data=yearly, x='year', y='count')
plt.axhline(avg, color='red', linestyle='--', label=f'Average ({avg:.0f})')
plt.legend()
plt.title("Number of Cars per Year")
plt.show()

* An average of **225 cars per year** for our analysis

##### *Periodic Trends Analysis (Monthly - Yearly - Quarterly)*

In [ ]:
monthly_df = df[['price', 'month']].groupby('month').aggregate('mean')
monthly_df = monthly_df.apply(lambda x: round(x))

yearly_df = df[['price', 'year']].groupby('year').aggregate('mean')
yearly_df = yearly_df.apply(lambda x: round(x))

quarterly_df = df[['price', 'quarter']].groupby('quarter').aggregate('mean')
quarterly_df = quarterly_df.apply(lambda x: round(x))

fig, axes = plt.subplots(1, 3, figsize=(18,5))

sns.lineplot(data=monthly_df, x='month', y='price', color='royalblue', ax=axes[0])
axes[0].set_title('Price Monthly Trend Analysis')
axes[0].tick_params(axis='x', rotation=45)

sns.lineplot(data=quarterly_df, x='quarter', y='price', color='royalblue', ax=axes[1])
axes[1].set_title('Price Quarterly Trend Analysis')
axes[1].tick_params(axis='x', rotation=45)

sns.lineplot(data=yearly_df, x='year', y='price', color='royalblue', ax=axes[2])
axes[2].set_title('Price Yearly Trend Analysis')
axes[2].tick_params(axis='x', rotation=45)

plt.suptitle('Periodic Price Trend Analysis', fontsize=14)
plt.tight_layout()
plt.show()

* The *monthly* chart shows two price spikes, one in **June** and another in **December**.
* The *average yearly price* has been rising over the past 18 years, peaking after the **COVID-19** lockdown.

## **Bivariate Analysis** 

##### *Correlation Heatmap*

In [ ]:
fig = plt.figure(figsize=(9,7))
correlation_matrix = df.corr(numeric_only=True)
sns.heatmap(correlation_matrix, cmap='coolwarm', annot=True, fmt=".2f", square=True, linewidths=.5)
plt.show()

* *Price* and *Fiscal Power* show moderate correlation, while *Car Age* and *Mileage* exhibit a stronger one
* The correlation between *Price* and *Year* is weak

##### *Average Price per Brand*

In [ ]:
temp1 = df.groupby('brand').agg(price=('price','mean'),frequency=('price','count')).sort_values(by='price', ascending=False).head(7).drop(['mg', 'renault truck'], errors='ignore')
temp2 = df.groupby('brand')[['price']].mean().sort_values(by='price', ascending=True).head(5)

fig, ax = plt.subplots(figsize=(10,6))

sns.barplot(
    data=temp1.reset_index(),
    x='brand', y='price',
    color='royalblue', ax=ax
)

sns.barplot(
    data=temp2.reset_index(),
    x='brand', y='price',
    color='red', ax=ax
)

for container in ax.containers[len(temp1):]:
    for bar in container:
        bar.set_height(-bar.get_height())

ax.set_title("Most Expensive vs Cheapest Brands", fontsize=14)
ax.set_ylabel("Average Price")
ax.set_xlabel("Brand")
ax.axhline(0, color="black", linewidth=0.8)
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

* Premium brands like *Porsche* and *Land Rover* record the highest average prices.
* The inclusion of *Chery* among the top five is due to its recent entry into the Tunisian market, resulting in **higher initial prices**.

##### *Locations With The Highest Prices*

In [ ]:
temp = df[['location','price']].groupby('location').aggregate('mean').sort_values(by='price', ascending=False).head(6)

plt.figure(figsize=(5,5))

sns.barplot(data=temp, x='location', y='price')

plt.ylabel('average price')
plt.xticks(rotation=45)
plt.title('Locations With The Highest Prices')
plt.tight_layout()
plt.show()

* Urban locations like *Sousse* and *Tunis* are taking the lead, followed by *Sidi Bouzid* and *Tatouine* with a slight difference.

##### *Average Mileage (small data available for this feature) & Average Price For Each Fuel Type*

In [ ]:
temp1 = df[['mileage','fuel']].groupby('fuel').aggregate('mean')
temp2 = df[['fuel','price']].groupby('fuel').aggregate('mean')

fig, axes = plt.subplots(1,2, figsize=(9,4))

sns.barplot(data=temp1, x='fuel', y='mileage', color='royalblue', ax=axes[0])
axes[0].set_title('Average Mileage per Fuel Type')

sns.barplot(data=temp2, x='fuel', y='price', color='royalblue', ax=axes[1])
axes[1].set_title('Average Mileage per Fuel Type')

plt.suptitle('Average Mileage & Average Price for Each Fuel Type')
plt.tight_layout()
plt.show()

* *Diesel* vehicles have higher **mileage** than *Petrol* ones, which aligns with expectations despite limited mileage data.
* There is almost no difference in **average prices** between *Petrol* and *Diesel* cars.

##### *Analyzing Monthly Price Distribution*

In [ ]:
import calendar

temp = df.copy()
temp['month'] = temp['month'].map({i: m for i, m in enumerate(calendar.month_name) if i})
df['month'] = df['month'].map({i: m for i, m in enumerate(calendar.month_name) if i})

temp = temp[['month', 'price']].groupby('month', as_index=False).mean()
temp['price'] = temp['price'].round()

order = list(calendar.month_name)[1:]
temp['month'] = pd.Categorical(temp['month'], categories=order, ordered=True)
temp = temp.sort_values('month').reset_index(drop=True)

fig, axes = plt.subplots(2, 6, figsize=(18.75, 8))

for idx, month in enumerate(order):
    sns.boxplot(
        y=df[df['month'] == month]['price'],  
        ax=axes.flatten()[idx],
        color='royalblue'
    )
    axes.flatten()[idx].set_title(month)

plt.suptitle('The Distribution Of Prices per Month', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

* **May** and **October** show wider distributions and higher outliers, *suggesting greater price variability.*

##### *The Evolution Of Fuel Types*

In [ ]:
temp = pd.crosstab(df['year'], df['fuel']).reset_index()

plt.figure(figsize=(10, 6))
sns.lineplot(data=temp, x='year', y='diesel', label='Diesel', marker="o")
sns.lineplot(data=temp, x='year', y='essence', label='Essence', marker="o")

plt.title("Fuel Type Counts per Year")
plt.xlabel("Year")
plt.ylabel("Count")
plt.xticks(temp['year'])
plt.legend()
plt.grid(True)
plt.show()


* The trend shows that *Petrol* cars have consistently been more common than *Diesel* cars.

## **Multivariate Analysis** 

##### *Does Higher Mileage Cause Lower Prices ?*

In [ ]:
temp = df[(df['price']<100000) & (df['mileage']<250000)]

fig, axes = plt.subplots(1,2, figsize=(7,5))

sns.scatterplot(data=df, x='mileage', y='price', hue='year', color='royalblue', ax=axes[0])
axes[0].set_title('Mileage Vs. Price (Full View)')

sns.scatterplot(data=temp, x='mileage', y='price', hue='year', color='red', ax=axes[1])
axes[1].set_title('Mileage Vs. Price (Zoomed In)')

plt.suptitle('Mileage Vs. Price')
plt.tight_layout()
plt.show()

* In the early years (2008, 2011), data points are more spread out, indicating a **stronger market** and **weak effect of mileage on price**.

* A clear *downward trend* suggests that price is consistently affected by *mileage* across all periods.

* *High-mileage* cars from the *last five years* cluster with *low-mileage* cars from *2011 and earlier*, indicating **economic struggles in Tunisia and the significant impact of inflation**.

* *Recent cars* show a strong negative correlation between Mileage and Price, unlike *older models* that maintained more consistent pricing.

##### *Yearly Evolution of Car Prices by Fuel Type*

In [ ]:
temp = df.groupby(['year','fuel'])['price'].mean().reset_index()

plt.figure(figsize=(8,4))
sns.lineplot(data=temp, x='year', y='price', hue='fuel', marker='o')

plt.title('Average Yearly Price per Fuel Type')
plt.xlabel('Year')
plt.xticks(sorted(df['year'].unique()), rotation=45)
plt.ylabel('Average Price')
plt.show()

* The graph shows a **consistent upward trend**, with *Diesel* prices outperforming *Petrol* prices over time.

* A noticeable price spike in 2021 likely reflects *post-COVID* economic effects. *(supported by online reports)*.

##### *Average Fuel Price by Brand For the Most Common Car Brands.*

In [ ]:
most_commun_brands = df['brand'].value_counts().head(7).index.tolist()
temp = df.groupby(['brand', 'fuel'])['price'].mean().reset_index()
temp = temp[temp['brand'].isin(most_commun_brands)]

plt.figure(figsize=(8,4))
sns.barplot(data=temp, x='brand', y='price', hue='fuel')
plt.tight_layout()
plt.title('Average Fuel Price by Brand')
plt.xlabel('Top 6 Brands by Frequency')
plt.ylabel('Average Price')
plt.xticks(rotation=45)
plt.show()

* **Renault** and **Volkswagen** show similar average prices, suggesting balanced markets between the two fuel types, *possibly influenced by commercial vehicles.*

* **Ford** and **BMW** display a clear price gap, with petrol models leading, *reflecting consumer preference for petrol in non-commercial cars.*

* Overall, **petrol** cars tend to be more expensive than **diesel** ones across most brands, *likely due to higher demand or better reliability of petrol models.*

##### *Most Common Models of the Top Car Brands.*

In [ ]:
most_commun_brands = df['brand'].value_counts().head(7).index.tolist()
temp = df[df['brand'].isin(most_commun_brands)]
top3_models = (
    temp.groupby(['brand', 'model'])
      .size()
      .reset_index(name='count')
      .sort_values(['brand', 'count'], ascending=[True, False])
      .groupby('brand')
      .head(3)
)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Top 3 Most Frequent Models per Brand', fontsize=16, fontweight='bold', y=0.995)

brands = top3_models['brand'].unique()

for idx, brand in enumerate(brands):
    row = idx // 4
    col = idx % 4
    ax = axes[row, col]
    
    brand_data = top3_models[top3_models['brand'] == brand]
    
    # Create bar plot
    sns.barplot(
        data=brand_data, 
        x='model', 
        y='count', 
        ax=ax,
        palette='viridis',
        edgecolor='black',
        linewidth=0.5
    )
    
    ax.set_title(brand.capitalize(), fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Number of Listings' if col == 0 else '')
    ax.tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for container in ax.containers:
        ax.bar_label(container, fmt='%d', padding=3)

plt.tight_layout()
plt.show()

* **Small cars** dominate the Tunisian market, with models like the BMW Series 3, Peugeot 206, and Renault Clio leading sales.

## **CONCLUSION** 

* Since 2021, the available data volume has declined due to limited records from that period, which may introduce bias in frequency-based analyses. However, based on domain knowledge, the dataset remains sufficient to provide a reliable market overview over time.

* The absence of around 80% of mileage values does not affect the analysis, as mileage is not a key factor and was not used for predictive modeling.

* The exploratory analysis of the 2007–2025 dataset (4,500 records) shows that small cars—such as the BMW Series 3, Peugeot 206, and Renault Clio—dominate the Tunisian market, reflecting consumer preference for compact, affordable vehicles among the top seven brands.

* Nominal prices have generally risen over time, with a notable 50% increase in petrol car prices in 2021 compared to diesel. Although this may reflect a 4:1 data imbalance favoring petrol listings, domain knowledge and supporting articles confirm the trend’s validity.

* A slight price increase after the 2011 Tunisian Revolution suggests the decrease of the local currency against the dollar, can be tied to an increase in demand.